# Lesson 2: Tools, Tool Calls, and the Agent Execution Loop

This notebook covers:
1. Creating a custom tool and observing the full message trace (`HumanMessage` -> `AIMessage (tool_call)` -> `ToolMessage` -> `AIMessage`).
2. Testing bad vs. good tool descriptions.
3. Finding the model's knowledge cutoff hallucination and fixing it with a search tool.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

# Load environment variables (GROQ_API_KEY, TAVILY_API_KEY)
load_dotenv()

# Initialize LLM
llm = ChatGroq(
    model_name="qwen/qwen3.8-27b",
    temperature=0.0
)

# Define a simple custom tool
@tool
def calculate_shipping_cost(weight_kg: float, distance_km: float) -> str:
    """Calculates the shipping cost in USD based on package weight in kg and distance in km."""
    base_rate = 5.0
    cost = base_rate + (weight_kg * 1.5) + (distance_km * 0.1)
    return f"The total shipping cost is ${cost:.2f} USD."

In [3]:
# Create agent with tool
agent = create_agent(
    model=llm,
    tools=[calculate_shipping_cost],
    system_prompt="You are a helpful logistics assistant. Always use tools to calculate shipping costs."
)

# Run query that triggers the tool
query = "How much will it cost to ship a 4 kg package over 120 km?"
response = agent.invoke({"messages": [HumanMessage(content=query)]})

print("=" * 70)
print("TASK 1: FULL MESSAGE TRACE & EMPTY AI MESSAGE WITH TOOL CALL")
print("=" * 70)

for i, msg in enumerate(response["messages"]):
    print(f"\n--- MESSAGE {i+1}: [{msg.type.upper()}] ---")
    print(f"Content: {repr(msg.content)}")
    
    # Highlight tool calls inside the AIMessage
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"\n TOOL CALLS FOUND: {msg.tool_calls}")

TASK 1: FULL MESSAGE TRACE & EMPTY AI MESSAGE WITH TOOL CALL

--- MESSAGE 1: [HUMAN] ---
Content: 'How much will it cost to ship a 4 kg package over 120 km?'

--- MESSAGE 2: [AI] ---
Content: ''

 TOOL CALLS FOUND: [{'name': 'calculate_shipping_cost', 'args': {'distance_km': 120, 'weight_kg': 4}, 'id': 'ffcr3xckp', 'type': 'tool_call'}]

--- MESSAGE 3: [TOOL] ---
Content: 'The total shipping cost is $23.00 USD.'

--- MESSAGE 4: [AI] ---
Content: 'It will cost **$23.00 USD** to ship a 4 kg package over a distance of 120 km.'


In [6]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

#  BAD TOOL DESCRIPTION 

@tool
def do_thing(a: float, b: float) -> str:
    """Stuff."""
    return f"Result: {a * b}"

agent_bad = create_agent(
    model=llm,
    tools=[do_thing],
    system_prompt="You are a helpful assistant. Use tools when needed."
)

query = "Calculate the total cost of 5 items priced at $12 each."

print("=" * 70)
print("--- 1. BAD TOOL DESCRIPTION  ---")
print("=" * 70)
bad_response = agent_bad.invoke({"messages": [HumanMessage(content=query)]})

for msg in bad_response["messages"]:
    print(f"[{msg.type.upper()}]: {msg.content}")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"   Tool Call: {msg.tool_calls}")


#  GOOD TOOL DESCRIPTION 

@tool
def calculate_total_price(quantity: float, unit_price: float) -> str:
    """Calculates the total cost by multiplying the quantity of items by their unit price."""
    return f"The total price is ${quantity * unit_price:.2f}"

agent_good = create_agent(
    model=llm,
    tools=[calculate_total_price],
    system_prompt="You are a helpful assistant. Always use tools for calculations."
)

print("\n" + "=" * 70)
print("--- 2. GOOD TOOL DESCRIPTION  ---")
print("=" * 70)
good_response = agent_good.invoke({"messages": [HumanMessage(content=query)]})

for msg in good_response["messages"]:
    print(f"[{msg.type.upper()}]: {msg.content}")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"   Tool Call: {msg.tool_calls}")

--- 1. BAD TOOL DESCRIPTION  ---
[HUMAN]: Calculate the total cost of 5 items priced at $12 each.
[AI]: 
   Tool Call: [{'name': 'do_thing', 'args': {'a': 5, 'b': 12}, 'id': 'wx65731k4', 'type': 'tool_call'}]
[TOOL]: Result: 60.0
[AI]: The total cost of 5 items priced at $12 each is **$60**.

--- 2. GOOD TOOL DESCRIPTION  ---
[HUMAN]: Calculate the total cost of 5 items priced at $12 each.
[AI]: 
   Tool Call: [{'name': 'calculate_total_price', 'args': {'quantity': 5, 'unit_price': 12}, 'id': 'gda8jhyz8', 'type': 'tool_call'}]
[TOOL]: The total price is $60.00
[AI]: The total cost of 5 items priced at $12 each is **$60.00**.


In [9]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

#  BAD TOOL DESCRIPTION
@tool
def process(x: float, y: float) -> str:
    """Processes x and y."""
    discounted = x - (x * (y / 100))
    return f"Final: ${discounted:.2f}"

agent_bad = create_agent(
    model=llm,
    tools=[process],
    system_prompt="You are a retail shopping assistant."
)

query = "Apply a 20% discount to a $150 jacket."

print("=" * 70)
print("--- 1. BAD TOOL DESCRIPTION ---")
print("=" * 70)
bad_response = agent_bad.invoke({"messages": [HumanMessage(content=query)]})

for msg in bad_response["messages"]:
    print(f"[{msg.type.upper()}]: {msg.content}")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"   Tool Call: {msg.tool_calls}")


#  GOOD TOOL DESCRIPTION
@tool
def calculate_discounted_price(original_price: float, discount_percentage: float) -> str:
    """Calculates the final price after applying a percentage discount to the original item price."""
    discounted = original_price - (original_price * (discount_percentage / 100))
    return f"The discounted price is ${discounted:.2f}"

agent_good = create_agent(
    model=llm,
    tools=[calculate_discounted_price],
    system_prompt="You are a retail shopping assistant."
)

print("\n" + "=" * 70)
print("--- 2. GOOD TOOL DESCRIPTION ---")
print("=" * 70)
good_response = agent_good.invoke({"messages": [HumanMessage(content=query)]})

for msg in good_response["messages"]:
    print(f"[{msg.type.upper()}]: {msg.content}")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"   Tool Call: {msg.tool_calls}")

--- 1. BAD TOOL DESCRIPTION ---


[HUMAN]: Apply a 20% discount to a $150 jacket.
[AI]: 
   Tool Call: [{'name': 'process', 'args': {'x': 150, 'y': 20}, 'id': 'mam4r2ppz', 'type': 'tool_call'}]
[TOOL]: Final: $120.00
[AI]: After applying a 20% discount to the $150 jacket, the final price is **$120.00**.

--- 2. GOOD TOOL DESCRIPTION ---
[HUMAN]: Apply a 20% discount to a $150 jacket.
[AI]: 
   Tool Call: [{'name': 'calculate_discounted_price', 'args': {'discount_percentage': 20, 'original_price': 150}, 'id': '5v666jgj2', 'type': 'tool_call'}]
[TOOL]: The discounted price is $120.00
[AI]: After applying a 20% discount to the $150 jacket, the final price is **$120.00**.


In [19]:
import os
from dotenv import load_dotenv
from langchain_community.tools import TavilySearchResults
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

load_dotenv(override=True)

# Extract Tavily API Key
tavily_key = os.getenv("TAVILY_API_KEY")
if not tavily_key:
    raise ValueError("TAVILY_API_KEY is missing! Check your .env file.")

question = "Who won the FIFA World Cup 2026 final match and what was the score?"

#  MODEL WITHOUT TOOL 
no_tool_agent = create_agent(model=llm, tools=[])
no_tool_res = no_tool_agent.invoke({"messages": [HumanMessage(content=question)]})

print("=" * 70)
print(" MODEL WITHOUT TOOL")
print("=" * 70)
print(no_tool_res["messages"][-1].content)


#  MODEL WITH SEARCH TOOL 
search_tool = TavilySearchResults(max_results=3, tavily_api_key=tavily_key)

search_agent = create_agent(
    model=llm,
    tools=[search_tool],
    system_prompt=(
        "You are a real-time factual assistant operating in 2026. "
        "You MUST invoke the search tool to verify live match results and historical sports scores before generating any response."
    )
)

search_res = search_agent.invoke({"messages": [HumanMessage(content=question)]})

print("\n" + "=" * 70)
print("MODEL WITH SEARCH TOOL ")
print("=" * 70)

for msg in search_res["messages"]:
    print(f"[{msg.type.upper()}]: {msg.content}")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"  Tool Call Triggered: {msg.tool_calls}")

 MODEL WITHOUT TOOL
The **2026 FIFA World Cup has not yet taken place**.

The tournament is scheduled to be held in **June and July 2026** across the United States, Canada, and Mexico. Therefore, there is no final match, winner, or score to report at this time.

### Key Facts:
- **Hosts:** United States, Canada, and Mexico (the first World Cup hosted by three countries).
- **Format:** Expanded to 48 teams (up from 32).
- **Final Match Date:** Scheduled for **July 19, 2026**, at MetLife Stadium in New Jersey, USA.

You can follow official updates from [FIFA.com](https://www.fifa.com) for the latest news, draw results, and match schedules as the tournament approaches.

MODEL WITH SEARCH TOOL 
[HUMAN]: Who won the FIFA World Cup 2026 final match and what was the score?
[AI]: 
  Tool Call Triggered: [{'name': 'tavily_search_results_json', 'args': {'query': 'FIFA World Cup 2026 final match winner score'}, 'id': '3y5w4cgxw', 'type': 'tool_call'}]
[TOOL]: [{"title": "2026 FIFA World Cup final